# Cell3 Internet Traffic EDA

This notebook explores the `Cell3_Data` internet traffic files and focuses on one busy `CellID`: the cell with the highest mean internet traffic across the full seven-day period.

Main goals:
- load and combine the seven daily Cell3 files
- identify the busiest cell by weekly mean internet traffic
- visualize the full 7-day traffic pattern for that cell
- summarize average internet traffic by hour of day and by day of week
- save the three-chart figure to `notebooks/cell3_eda.png`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


## 1. Load and prepare the Cell3 dataset

The raw files can contain multiple rows for the same `CellID` and timestamp, so internet traffic is aggregated at the cell-timestamp level before analysis.

In [ ]:
cell3_files = sorted(Path("../data/Cell3_Data").glob("sms-call-internet-mi-2013-11-*.csv"))
cell3_df = pd.concat((pd.read_csv(path, usecols=["CellID", "datetime", "internet"]) for path in cell3_files), ignore_index=True)
cell3_df = cell3_df.dropna(subset=["internet"]).copy()
cell3_df = cell3_df.groupby(["CellID", "datetime"], as_index=False)["internet"].sum()
cell3_df["datetime"] = pd.to_datetime(cell3_df["datetime"], unit="ms")

print("Files loaded:", len(cell3_files))
print("Combined shape:", cell3_df.shape)
cell3_df.head()


Insight:
- Aggregating by `CellID` and `datetime` gives one internet traffic value per cell per time step.
- This is the right level for identifying the busiest cell across the week.

## 2. Find the busiest cell across the week

The busiest cell is defined as the `CellID` with the highest mean internet traffic over the full seven days.

In [ ]:
cell_mean_traffic = cell3_df.groupby("CellID")["internet"].mean().sort_values(ascending=False)
busy_cell_id = int(cell_mean_traffic.idxmax())
busy_cell_df = cell3_df[cell3_df["CellID"] == busy_cell_id].sort_values("datetime").copy()
busy_cell_df["hour"] = busy_cell_df["datetime"].dt.hour
busy_cell_df["day_of_week"] = busy_cell_df["datetime"].dt.dayofweek

print(f"Busiest CellID: {busy_cell_id}")
print(f"Weekly mean internet traffic: {cell_mean_traffic.iloc[0]:.2f}")
cell_mean_traffic.head(10)


Insight:
- This identifies a single reference cell for detailed EDA.
- In this dataset, the busiest cell is expected to be `CellID 5161` when the data is aggregated at the cell-timestamp level.

## 3. Internet traffic patterns for the busiest cell

The figure below contains three charts:
- internet traffic over the full 7 days
- average internet traffic by hour of day
- average internet traffic by day of week


In [ ]:
hourly_avg = busy_cell_df.groupby("hour")["internet"].mean().reindex(range(24))
daily_avg = busy_cell_df.groupby("day_of_week")["internet"].mean().reindex(range(7))
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, axes = plt.subplots(3, 1, figsize=(14, 16))

axes[0].plot(busy_cell_df["datetime"], busy_cell_df["internet"], color="navy", linewidth=1)
axes[0].set_title(f"Cell {busy_cell_id} Internet Traffic Across 7 Days")
axes[0].set_xlabel("datetime")
axes[0].set_ylabel("internet")

axes[1].bar(hourly_avg.index, hourly_avg.values, color="steelblue")
axes[1].set_title(f"Cell {busy_cell_id} Average Internet Traffic by Hour of Day")
axes[1].set_xlabel("hour")
axes[1].set_ylabel("mean internet")
axes[1].set_xticks(range(24))

axes[2].bar(day_labels, daily_avg.values, color="darkorange")
axes[2].set_title(f"Cell {busy_cell_id} Average Internet Traffic by Day of Week")
axes[2].set_xlabel("day_of_week")
axes[2].set_ylabel("mean internet")

output_path = Path("cell3_eda.png")
plt.tight_layout()
plt.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {output_path.resolve()}")


Insight:
- The full time-series plot shows how traffic rises and falls across the entire week.
- The hourly bar chart helps identify recurring busy hours.
- The day-of-week chart shows whether demand is stronger on weekdays or weekends.